stall_native_kaggle.py
Run STALL (Ben Hayun et al., CVPR 2026) with its OFFICIAL code and DEFAULT
settings on FakeParts and ViF-Bench, subset by subset, on Kaggle.

Everything that decides the score comes from the official repository
  github.com/OmerBenHayun/STALL  (pinned commit below):
  - frame selection   video_index.get_video_metadata / downsample_frames /
                      compute_windows  (8 fps, 2-s window = 16 frames)
  - model + scoring   stall.STALL  (DINOv3 ViT-L/16, spatial max / temporal min
                      log-likelihood, percentile vs. calibration, average)
  - calibration       precomputed/stall_params_vatex_dino_v3.npz
                      (33,976 real VATEX videos, shipped with the code)
This notebook only loops over your subsets and saves one CSV per subset.

Each "# %%" block is one Kaggle cell. Turn on a GPU and Internet.
  Cell 1  settings          -> edit DATASET, SUBSET_NAME, SUBSET_DIR, LABEL
  Cell 2  setup + functions -> once per session (needs the DINOv3 weights, see below)
  Cell 3  score this subset
  Cell 4  (optional) delete the subset's videos
  ...repeat cells 1, 3, 4 for every subset...
  Cell 5  final metrics     -> once, after all subsets

DINOv3 WEIGHTS (needed once): the official code loads the file
  dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
Meta distributes it after a short request form:
  https://ai.meta.com/resources/models-and-libraries/dinov3-downloads/
You receive a download link by e-mail. Either paste that link into
DINO_WEIGHTS_URL, or upload the .pth file as a private Kaggle dataset and set
DINO_WEIGHTS_PATH to it (recommended: the e-mailed link expires).

In [ ]:
# %% --------------------------------------------------------- CELL 1: SETTINGS
DATASET     = "FakeParts"                 # "FakeParts" or "ViF-Bench"
SUBSET_NAME = "t2v"                       # short name; used as category in the results
SUBSET_DIR  = "/kaggle/working/data/t2v"  # folder with this subset's videos
LABEL       = 1                           # 0 = real, 1 = generated / edited

DURATION    = 2          # STALL default: 2-s window at 8 fps = 16 frames (1 = sensitivity run)
VIDEO_BATCH = 8          # videos per GPU pass (official default)
OUT_ROOT    = "/kaggle/working/stall_native"

DINO_WEIGHTS_PATH = "/kaggle/input/dinov3-weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth"
DINO_WEIGHTS_URL  = None                  # or the link from Meta's e-mail

def VIDEO_ID(rel_path_without_ext):       # must match the ids of your own scores
    return f"{SUBSET_NAME}/{rel_path_without_ext}"

In [ ]:
# %% ------------------------------------------------- CELL 2: SETUP + FUNCTIONS
import os, sys, subprocess, json, time, csv, shutil
from pathlib import Path

STALL_REPO   = "/kaggle/working/STALL"
STALL_COMMIT = "f6dbb617a08d522f42ac02341028da8c58e669d1"   # official code, 25 Oct 2026
DINO_REPO    = "/kaggle/working/dinov3"

if not Path(STALL_REPO).exists():
    subprocess.run(["git", "clone", "https://github.com/OmerBenHayun/STALL", STALL_REPO], check=True)
subprocess.run(["git", "-C", STALL_REPO, "checkout", "-q", STALL_COMMIT], check=True)
if not Path(DINO_REPO).exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/facebookresearch/dinov3", DINO_REPO], check=True)
if shutil.which("ffprobe") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)

if not Path(DINO_WEIGHTS_PATH).exists():
    if not DINO_WEIGHTS_URL:
        raise FileNotFoundError("DINOv3 weights not found: set DINO_WEIGHTS_PATH or DINO_WEIGHTS_URL")
    DINO_WEIGHTS_PATH = "/kaggle/working/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth"
    if not Path(DINO_WEIGHTS_PATH).exists():
        subprocess.run(["wget", "-q", "-O", DINO_WEIGHTS_PATH, DINO_WEIGHTS_URL], check=True)

sys.path.insert(0, str(Path(STALL_REPO) / "src"))
import numpy as np
import torch
from stall import STALL                                                   # official
from video_index import get_video_metadata, downsample_frames, compute_windows  # official

PARAMS = np.load(Path(STALL_REPO) / "precomputed" / "stall_params_vatex_dino_v3.npz")
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v", ".gif"}
FIELDS = ["video_id", "subset", "label", "relpath", "fps", "duration_s", "n_frames_8fps",
          "spat_percentile", "temp_percentile", "stall_score", "score", "status"]
TARGET_FPS = 8

_det = None
def get_detector():
    global _det
    if _det is None:
        _det = STALL("cuda" if torch.cuda.is_available() else "cpu", dict(PARAMS),
                     dino_repo=DINO_REPO, dino_weights=DINO_WEIGHTS_PATH)
    return _det


def frame_window(path):
    """Official selection: skip fps < 8 or duration < 1 s; downsample to 8 fps;
    pick a DURATION-second window with the official seeded RNG."""
    md = get_video_metadata(str(path))
    fps, dur, n = md["fps"], md["duration_seconds"], md["num_frames"]
    if fps is None or dur is None:
        return md, None, "skipped_probe_failed", 0
    if fps < TARGET_FPS:
        return md, None, "skipped_fps_below_8", 0
    if dur < 1.0:
        return md, None, "skipped_shorter_than_1s", 0
    idxs = downsample_frames(int(n), fps, TARGET_FPS)
    win = compute_windows(idxs, target_fps=TARGET_FPS)[f"{DURATION}_sec_idxs"]
    if win is None:
        return md, None, f"skipped_shorter_than_{DURATION}s_window", len(idxs)
    return md, win, "ok", len(idxs)


def score_subset():
    root = Path(SUBSET_DIR)
    files = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
    if not files:
        raise RuntimeError(f"No videos under {root}")
    out_csv = Path(OUT_ROOT) / DATASET / f"{SUBSET_NAME}.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    done = set()
    if out_csv.exists():                                   # resume after interruption
        with open(out_csv) as f:
            done = {r["video_id"] for r in csv.DictReader(f)}
    todo = [(p, p.relative_to(root)) for p in files
            if VIDEO_ID(str(p.relative_to(root).with_suffix(""))) not in done]
    print(f"[{DATASET}/{SUBSET_NAME}] {len(files)} videos, {len(done)} already scored, {len(todo)} to do")

    det = get_detector()
    fh = open(out_csv, "a", newline="")
    w = csv.DictWriter(fh, fieldnames=FIELDS)
    if not done:
        w.writeheader()

    batch, t0 = [], time.time()
    def flush(batch):
        if not batch:
            return
        try:
            res = det.batch_inference([str(b[0]) for b in batch], [b[2] for b in batch])
        except Exception as e:                             # fall back to one by one
            res = []
            for b in batch:
                try:
                    res.append(det.inference(str(b[0]), b[2]))
                except Exception as e2:
                    res.append(e2)
        for (p, row, _), r in zip(batch, res):
            if isinstance(r, Exception):
                row["status"] = f"error: {str(r)[:120]}"
            else:
                s = float(r["final_score"][0])             # higher = more likely REAL
                row.update(spat_percentile=float(r["spat_percentile"][0]),
                           temp_percentile=float(r["temp_percentile"][0]),
                           stall_score=s, score=1.0 - s, status="ok")   # score: higher = generated
            w.writerow(row)
        fh.flush()

    for k, (p, rel) in enumerate(todo, 1):
        vid = VIDEO_ID(str(rel.with_suffix("")))
        md, win, status, n8 = frame_window(p)
        row = dict(video_id=vid, subset=SUBSET_NAME, label=LABEL, relpath=str(rel),
                   fps=md.get("fps"), duration_s=md.get("duration_seconds"), n_frames_8fps=n8,
                   spat_percentile="", temp_percentile="", stall_score="", score="", status=status)
        if status != "ok":
            w.writerow(row)
        else:
            batch.append((p, row, win))
            if len(batch) == VIDEO_BATCH:
                flush(batch); batch = []
        if k % 200 == 0 or k == len(todo):
            print(f"  {k}/{len(todo)}  ({(time.time() - t0) / k:.2f} s/video)", flush=True)
    flush(batch)
    fh.close()
    print("Saved", out_csv)


# ------------------------------------------------------------------ metrics
def _auroc(real, fake):
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(np.r_[np.zeros(len(real)), np.ones(len(fake))], np.r_[real, fake])


def _recall_at_fpr(real, fake, fpr):
    thr = np.quantile(real, 1 - fpr, method="higher")
    return float((fake >= thr).mean())


def _metrics(real, fake):
    from sklearn.metrics import average_precision_score
    y = np.r_[np.zeros(len(real)), np.ones(len(fake))]
    return {"n_real": len(real), "n_fake": len(fake),
            "AUROC": 100 * _auroc(real, fake),
            "AP_fake": 100 * average_precision_score(y, np.r_[real, fake]),
            "recall@1%FPR": 100 * _recall_at_fpr(real, fake, 0.01),
            "recall@5%FPR": 100 * _recall_at_fpr(real, fake, 0.05)}


def final_metrics(primary_csv=None, primary_id_col="video_id", primary_score_col="score"):
    """Pooled and per-subset metrics on ALL scored videos (generated = positive);
    optionally complementarity with your primary cross-encoder score."""
    import pandas as pd
    d = Path(OUT_ROOT) / DATASET
    df = pd.concat([pd.read_csv(p) for p in sorted(d.glob("*.csv")) if p.name != "SUMMARY.csv"],
                   ignore_index=True)
    print("Status counts (videos STALL's protocol cannot score are listed here):")
    print(df.groupby(["subset", "status"]).size().to_string(), "\n")
    ok = df[df.status == "ok"].copy()
    real = ok[ok.label == 0].score.to_numpy()
    rows = [{"group": "ALL", **_metrics(real, ok[ok.label == 1].score.to_numpy())}]
    for sub, g in ok[ok.label == 1].groupby("subset"):
        rows.append({"group": sub, **_metrics(real, g.score.to_numpy())})
    res = pd.DataFrame(rows).round(2)
    print(res.to_string(index=False))

    if primary_csv:
        from scipy.stats import spearmanr
        pr = pd.read_csv(primary_csv)[[primary_id_col, primary_score_col]]
        pr.columns = ["video_id", "primary"]
        m = ok.merge(pr, on="video_id", how="inner")
        print(f"\nMerged with primary score: {len(m)} of {len(ok)} videos")
        ref = m[m.label == 0]
        pct = lambda v, r: np.searchsorted(np.sort(r), v, side="right") / len(r)
        m["comb"] = (pct(m.score, ref.score) + pct(m.primary, ref.primary)) / 2
        rr, ff = m[m.label == 0], m[m.label == 1]
        comp = {"spearman_rho": spearmanr(m.score, m.primary)[0],
                "AUROC_STALL": 100 * _auroc(rr.score, ff.score),
                "AUROC_primary": 100 * _auroc(rr.primary, ff.primary),
                "AUROC_combined": 100 * _auroc(rr.comb, ff.comb)}
        print({k: round(float(v), 3) for k, v in comp.items()})
        res = pd.concat([res, pd.DataFrame([{"group": "complementarity", **comp}])])
    res.to_csv(d / "SUMMARY.csv", index=False)
    print("\nSaved", d / "SUMMARY.csv")
    return res


print("STALL", STALL_COMMIT[:8], "| calibration: VATEX,", PARAMS["calib_ll_spat"].shape[0], "real videos")

In [ ]:
# %% ------------------------------------------------- CELL 3: SCORE THIS SUBSET
score_subset()

In [ ]:
# %% ---------------------------------------- CELL 4 (optional): DELETE SUBSET
# shutil.rmtree(SUBSET_DIR); print("Deleted", SUBSET_DIR)

In [ ]:
# %% --------------------------------- CELL 5: FINAL METRICS (after all subsets)
# final_metrics()
# final_metrics(primary_csv="/kaggle/input/<your-data>/primary_scores.csv",
#               primary_id_col="video_id", primary_score_col="cka")